# PRCP-1015 — Earthquake Damage Prediction
## Richter's Predictor: Modeling Earthquake Damage

**B.Tech AI/ML Capstone Project | Disaster Management / Structural Engineering**

**Project ID:** PRCP-1015  
**Target:** `damage_grade` (ordinal classes 1, 2, 3)

### Abstract
This notebook develops an end-to-end machine-learning workflow for predicting earthquake building damage from structural, geographic, ownership, and usage attributes. The dataset is the DrivenData *Richter's Predictor: Modeling Earthquake Damage* dataset associated with the 2015 Gorkha earthquake in Nepal.

The project covers dataset verification, exploratory data analysis, data-quality checks, leakage-aware preprocessing, feature engineering, multiple model development, evaluation using macro F1 and other classification metrics, feature-importance analysis, model comparison, challenges, and engineering-oriented recommendations. The notebook deliberately reports **measured results from the uploaded data** rather than hard-coding expected competition results.

> **Academic disclaimer:** This is an academic risk-assessment model, not a substitute for structural inspection, seismic engineering, building-code compliance, or emergency-management decisions.

## Table of Contents

1. Project objectives and problem statement
2. Dataset and source
3. Environment and configuration
4. Dataset loading and verification
5. Exploratory data analysis
6. Data quality and leakage analysis
7. Preprocessing and stratified splitting
8. Feature engineering
9. Model development
10. Model evaluation
11. Feature importance
12. Baseline vs engineered-feature analysis
13. Model comparison report
14. Challenges and techniques used
15. Suggestions for seismologists / disaster-management stakeholders
16. Limitations, future scope and conclusion
17. Reproducibility and execution guide

## 1. Problem Statement and Objectives

### Problem statement
Predict the ordinal `damage_grade` assigned to a building affected by the Gorkha earthquake:

- **1:** low damage
- **2:** medium damage
- **3:** almost complete destruction

Although the target is ordinal, this notebook evaluates the task primarily as a three-class classification problem because the project requirements specify multiclass predictive evaluation. Macro F1 is emphasized because it gives each damage grade equal weight.

### Objectives
- Verify the uploaded train-values and train-labels structure before modeling.
- Quantify class distribution, data quality, feature cardinality, and geographic structure.
- Explore structural and geographic relationships with damage grade.
- Build leakage-aware preprocessing and feature engineering.
- Compare an interpretable tree baseline, Random Forest, and LightGBM.
- Evaluate models using accuracy, macro/weighted precision, recall, F1, balanced accuracy, and confusion matrices.
- Identify influential features without interpreting obfuscated category letters as real-world semantic labels.
- Produce cautious, evidence-based suggestions for earthquake-risk reduction.

## 2. Dataset Description

The project source document describes 39 columns: one unique `building_id` plus 38 explanatory features. Geographic identifiers are available at three levels. Several construction and ownership variables are categorical, and superstructure/secondary-use fields are binary flags.

The source documentation states that the categorical values have been obfuscated. Therefore, this notebook **does not assign invented meanings to letters such as `r`, `u`, `w`, etc.** Any engineering recommendation is based on documented feature names, observed model associations, and domain caution—not on guessed category meanings.

Source: DrivenData, *Richter's Predictor: Modeling Earthquake Damage*.

In [ ]:
# 3. Imports and configuration
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix, classification_report,
    cohen_kappa_score
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "damage_grade"

# Set this to a folder containing the uploaded CSV files when running outside this environment.
DATA_DIR = Path(".")
TRAIN_VALUES_NAME = "train_values.csv"
TRAIN_LABELS_NAME = "train_labels.csv"

# Optional: if you have the original competition ZIP, put its path here.
DATASET_ZIP = None

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
print("Configuration ready.")

In [ ]:
# 4. Dataset discovery / loading
import zipfile

def safe_extract_zip(zip_path, extract_to):
    extract_to = Path(extract_to)
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        root = extract_to.resolve()
        for member in z.infolist():
            target = (extract_to / member.filename).resolve()
            if not str(target).startswith(str(root)):
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        z.extractall(extract_to)

def locate_file(name, base=Path(".")):
    candidates = [
        base / name,
        Path("/mnt/data") / name,
    ]
    # Search nearby folders, but avoid excessively broad filesystem traversal.
    for c in candidates:
        if c.exists():
            return c
    for root in [Path("."), Path("/mnt/data")]:
        if root.exists():
            hits = list(root.glob(f"**/{name}"))
            if hits:
                return hits[0]
    return None

if DATASET_ZIP:
    zip_path = Path(DATASET_ZIP)
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP not found: {zip_path}")
    extract_dir = Path("equake_dataset")
    safe_extract_zip(zip_path, extract_dir)

train_values_path = locate_file(TRAIN_VALUES_NAME, DATA_DIR)
train_labels_path = locate_file(TRAIN_LABELS_NAME, DATA_DIR)

# Support common alternative competition naming when files are renamed.
if train_values_path is None:
    alt = locate_file("train.csv", DATA_DIR)
    if alt is not None:
        train_values_path = alt

if train_values_path is None or train_labels_path is None:
    raise FileNotFoundError(
        "Could not find train_values.csv and train_labels.csv. "
        "Place them in DATA_DIR or set DATA_DIR to the dataset folder."
    )

values = pd.read_csv(train_values_path)
labels = pd.read_csv(train_labels_path)

print("Train values:", train_values_path)
print("Train labels:", train_labels_path)
print("Values shape:", values.shape)
print("Labels shape:", labels.shape)

# Separate train.csv-with-label format is also supported.
if TARGET in values.columns:
    data = values.copy()
else:
    if "building_id" not in values.columns or "building_id" not in labels.columns:
        raise ValueError("Both files must contain building_id for a safe merge.")
    data = values.merge(labels, on="building_id", how="inner", validate="one_to_one")

if TARGET not in data.columns:
    raise ValueError(f"Target column {TARGET!r} was not found after loading/merging.")

print("Final modeling dataset:", data.shape)
display(data.head())

In [ ]:
# 5. Structural verification
expected_feature_count = 39
print("Rows:", len(data))
print("Columns:", len(data.columns))
print("Duplicate rows:", data.duplicated().sum())
print("Unique building_id:", data["building_id"].nunique())
print("Missing cells:", int(data.isna().sum().sum()))
print("Target classes:", sorted(data[TARGET].dropna().unique().tolist()))

print("\nDtype counts:")
display(data.dtypes.value_counts().rename("count").to_frame())

print("\nTarget distribution:")
target_summary = (
    data[TARGET].value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)
target_summary["percentage"] = 100 * target_summary["count"] / len(data)
display(target_summary.round(2))

assert data["building_id"].is_unique, "building_id should be unique in train values."
assert set(data[TARGET].dropna().unique()).issubset({1,2,3}), "Unexpected target class."

### Verified uploaded-data snapshot

The uploaded files are expected to be verified by the preceding cell. In this project run, the uploaded `train_values.csv` contains **260,601 rows and 39 columns**, while `train_labels.csv` contains the corresponding building IDs and `damage_grade`. The observed target classes are 1, 2, and 3.

The actual class counts and percentages are calculated and displayed by the notebook rather than hard-coded.

In [ ]:
# 6. Feature-type inventory
feature_df = data.drop(columns=[TARGET])
categorical_cols = feature_df.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()

binary_cols = [
    c for c in numeric_cols
    if set(feature_df[c].dropna().unique()).issubset({0, 1})
]

continuous_or_count_cols = [c for c in numeric_cols if c not in binary_cols]

print("Categorical columns:", len(categorical_cols))
print(categorical_cols)
print("\nNumeric columns:", len(numeric_cols))
print("\nBinary numeric columns:", len(binary_cols))
print(binary_cols)
print("\nOther numeric/count columns:", continuous_or_count_cols)

print("\nCategorical cardinalities:")
display(pd.DataFrame({
    "column": categorical_cols,
    "unique_values": [data[c].nunique(dropna=True) for c in categorical_cols]
}).sort_values("unique_values", ascending=False))

print("\nGeographic cardinalities:")
display(data[["geo_level_1_id","geo_level_2_id","geo_level_3_id"]].nunique().rename("unique_values").to_frame())

## 5. Exploratory Data Analysis

Every plot below is paired with a short interpretation prompt. The exact visual pattern is calculated from the uploaded data at execution time.

Because categorical values are obfuscated, plots for those columns should be read as **category-code associations**, not as real-world labels.

In [ ]:
# 7. Target distribution
counts = data[TARGET].value_counts().sort_index().reindex([1,2,3], fill_value=0)
plt.figure(figsize=(7,4))
bars = plt.bar(counts.index.astype(str), counts.values)
plt.title("Damage Grade Distribution")
plt.xlabel("Damage grade")
plt.ylabel("Number of buildings")
plt.bar_label(bars, fmt="%d")
plt.tight_layout()
plt.show()

print("Interpretation: Compare the prevalence of the three damage grades. Macro F1 is used later so minority grades are not hidden by overall accuracy.")

In [ ]:
# 8. Numeric distributions by damage grade
plot_cols = ["age", "area_percentage", "height_percentage", "count_floors_pre_eq", "count_families"]
fig, axes = plt.subplots(len(plot_cols), 1, figsize=(9, 18))
for ax, col in zip(axes, plot_cols):
    sns.boxplot(data=data, x=TARGET, y=col, order=[1,2,3], ax=ax)
    ax.set_title(f"{col} by damage grade")
    ax.set_xlabel("Damage grade")
plt.tight_layout()
plt.show()

In [ ]:
# 9. Geographic damage summaries
geo_cols = ["geo_level_1_id", "geo_level_2_id", "geo_level_3_id"]
for col in geo_cols:
    summary = (
        data.groupby(col)[TARGET]
        .agg(["count", "mean"])
        .sort_values("mean", ascending=False)
        .head(15)
    )
    display(summary.style.set_caption(f"Top 15 {col} regions by mean damage grade"))

# Level 1 is low-cardinality and is especially suitable for a readable normalized chart.
geo1_ct = pd.crosstab(data["geo_level_1_id"], data[TARGET], normalize="index") * 100
geo1_ct = geo1_ct.sort_values(3, ascending=False).head(15)

geo1_ct.plot(kind="bar", stacked=True, figsize=(12,5))
plt.title("Top Level-1 Regions by Grade-3 Share")
plt.xlabel("geo_level_1_id")
plt.ylabel("Percentage within region")
plt.legend(title="Damage grade")
plt.tight_layout()
plt.show()

In [ ]:
# 10. Categorical features vs damage grade: normalized stacked bars
cat_for_plot = [
    "land_surface_condition", "foundation_type", "roof_type",
    "ground_floor_type", "other_floor_type", "position",
    "plan_configuration", "legal_ownership_status"
]
for col in cat_for_plot:
    ct = pd.crosstab(data[col], data[TARGET], normalize="index") * 100
    ct = ct.reindex(columns=[1,2,3], fill_value=0)
    ax = ct.plot(kind="bar", stacked=True, figsize=(8,4))
    ax.set_title(f"{col}: damage-grade composition")
    ax.set_xlabel(col)
    ax.set_ylabel("Percentage")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# 11. Superstructure prevalence
super_cols = [c for c in data.columns if c.startswith("has_superstructure_")]
super_prev = data[super_cols].mean().sort_values(ascending=False) * 100
display(super_prev.rename("prevalence_%").to_frame())

damage_super = data.groupby(TARGET)[super_cols].mean().T * 100
damage_super.plot(kind="bar", figsize=(14,6))
plt.title("Superstructure Flag Prevalence by Damage Grade")
plt.xlabel("Superstructure feature")
plt.ylabel("Percentage of buildings")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 12. Secondary-use prevalence
secondary_cols = [c for c in data.columns if c.startswith("has_secondary_use_")]
secondary_prev = data[secondary_cols].mean().sort_values(ascending=False) * 100
display(secondary_prev.rename("prevalence_%").to_frame())

damage_secondary = data.groupby(TARGET)[secondary_cols].mean().T * 100
damage_secondary.plot(kind="bar", figsize=(14,6))
plt.title("Secondary-Use Flag Prevalence by Damage Grade")
plt.xlabel("Secondary-use feature")
plt.ylabel("Percentage of buildings")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 13. Numeric correlation heatmap
numeric_for_corr = data.select_dtypes(include=[np.number]).drop(columns=[TARGET], errors="ignore")
corr = numeric_for_corr.corr()
plt.figure(figsize=(13,10))
sns.heatmap(corr, cmap="vlag", center=0)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()

print("Note: geographic IDs are identifiers for nested regions, not continuous physical measurements. Their Pearson correlations should therefore not be interpreted as ordinary physical correlations.")

## 6. Data Quality, Leakage and Limitations

Checks include missingness, duplicates, unique identifiers, rare categories, outlier ranges, and the leakage risk of using `building_id`.

`building_id` is excluded from modeling because it is a unique random identifier rather than a meaningful explanatory variable.

The three geographic IDs can be highly predictive. This is useful for this dataset but creates a generalization concern: a random row split can place buildings from the same geographic hierarchy in both training and test sets. Therefore, the notebook reports this limitation explicitly and does not claim that random-split performance represents performance on entirely unseen geographic regions.

In [ ]:
# 14. Data-quality report
missing = data.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Columns with missing values:")
display(missing.rename("missing_count").to_frame())

print("Duplicate building IDs:", data["building_id"].duplicated().sum())
print("Duplicate complete rows:", data.duplicated().sum())

# Rare categories
rare_rows = []
for c in categorical_cols:
    vc = data[c].value_counts(dropna=False)
    rare_rows.append(pd.DataFrame({
        "column": c,
        "category": vc.index.astype(str),
        "count": vc.values,
        "percentage": 100 * vc.values / len(data)
    }))
rare_table = pd.concat(rare_rows, ignore_index=True)
display(rare_table[rare_table["percentage"] < 1].sort_values(["column","percentage"]).head(50))

print("\nNumeric range summary:")
display(data[["age","area_percentage","height_percentage","count_floors_pre_eq","count_families"]].describe().T)

### Outlier interpretation

The presence of a large maximum age relative to the median is an example of a skewed numeric variable. Tree-based models are generally less sensitive to monotonic scaling and extreme numeric values than distance-based methods. The notebook therefore avoids unnecessary scaling for tree models and documents the observed ranges instead of deleting observations automatically.

Outliers are not removed blindly because extreme buildings may be legitimate and potentially important for damage prediction.

## 7. Train/Test Split and Leakage Prevention

A stratified 80/20 split is used so that the target distribution remains approximately comparable between training and evaluation sets.

For low-cardinality categorical variables in the scikit-learn tree models, one-hot encoding is fitted using the training partition only. LightGBM receives categorical columns as pandas categorical variables.

No target-based feature is computed from the full dataset. When frequency/target-derived geographic features are used, their mapping is fitted from the training data only.

In [ ]:
# 15. Stratified split
train_df, test_df = train_test_split(
    data,
    test_size=TEST_SIZE,
    stratify=data[TARGET],
    random_state=RANDOM_STATE
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print("\nTrain target proportions:")
display(train_df[TARGET].value_counts(normalize=True).sort_index().rename("proportion").to_frame())

print("\nTest target proportions:")
display(test_df[TARGET].value_counts(normalize=True).sort_index().rename("proportion").to_frame())

## 8. Feature Engineering

The following features are investigated because they have a direct relationship to the documented data structure:

- total number of superstructure material flags;
- total number of secondary-use flags;
- area × height interaction;
- floors × age interaction;
- combined geographic key;
- training-set frequency encoding for `geo_level_3_id`.

Frequency encoding is used instead of one-hot encoding for the high-cardinality level-3 geography. Importantly, its mapping is learned on the training partition only.

In [ ]:
# 16. Leakage-aware feature engineering
super_cols = [c for c in data.columns if c.startswith("has_superstructure_")]
secondary_cols = [c for c in data.columns if c.startswith("has_secondary_use_")]

def engineer_features(train_x, other_x=None):
    train_x = train_x.copy()
    other_x = train_x.copy() if other_x is None else other_x.copy()

    def add_basic(df):
        df["superstructure_count"] = df[super_cols].sum(axis=1)
        df["secondary_use_count"] = df[secondary_cols].sum(axis=1)
        df["area_height_interaction"] = df["area_percentage"] * df["height_percentage"]
        df["floors_age_interaction"] = df["count_floors_pre_eq"] * df["age"]
        df["geo12_key"] = (
            df["geo_level_1_id"].astype(str) + "_" +
            df["geo_level_2_id"].astype(str)
        )
        return df

    train_x = add_basic(train_x)
    other_x = add_basic(other_x)

    geo3_freq = train_x["geo_level_3_id"].value_counts(normalize=True)
    geo12_freq = train_x["geo12_key"].value_counts(normalize=True)

    train_x["geo3_frequency"] = train_x["geo_level_3_id"].map(geo3_freq).fillna(0)
    other_x["geo3_frequency"] = other_x["geo_level_3_id"].map(geo3_freq).fillna(0)

    train_x["geo12_frequency"] = train_x["geo12_key"].map(geo12_freq).fillna(0)
    other_x["geo12_frequency"] = other_x["geo12_key"].map(geo12_freq).fillna(0)

    return train_x, other_x

X_train_eng, X_test_eng = engineer_features(
    train_df.drop(columns=[TARGET, "building_id"]),
    test_df.drop(columns=[TARGET, "building_id"])
)

print("Engineered train shape:", X_train_eng.shape)
display(X_train_eng.head())

### Why not blindly use target encoding?

Target encoding can be powerful for high-cardinality geography, but it is especially vulnerable to target leakage. This notebook therefore uses a simpler training-only frequency encoding by default. If target encoding is added later, it should be computed inside cross-validation folds or an equivalent leakage-safe scheme.

## 9. Model Development

Three models are compared:

1. **Decision Tree** — interpretable baseline and useful for establishing a simple nonlinear benchmark.
2. **Random Forest** — bagged ensemble of decision trees, robust for mixed tabular features.
3. **LightGBM** — gradient-boosted decision trees designed for efficient large-scale tabular learning and able to work directly with categorical features.

The final selection is based on the measured metrics and computational observations produced by this notebook. No model is declared the production choice in advance.

In [ ]:
# 17. Prepare model matrices
X_train = train_df.drop(columns=[TARGET, "building_id"])
X_test = test_df.drop(columns=[TARGET, "building_id"])
y_train = train_df[TARGET]
y_test = test_df[TARGET]

low_card_cat = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# One-hot representation for Decision Tree / Random Forest.
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), low_card_cat)
    ],
    remainder="passthrough"
)

X_train_tree = tree_preprocessor.fit_transform(X_train)
X_test_tree = tree_preprocessor.transform(X_test)

print("Tree matrix shape:", X_train_tree.shape)

In [ ]:
# 18. Decision Tree
results = []
predictions = {}
fit_times = {}

start = time.perf_counter()
dt_model = DecisionTreeClassifier(
    max_depth=18,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE
)
dt_model.fit(X_train_tree, y_train)
dt_pred = dt_model.predict(X_test_tree)
fit_times["Decision Tree"] = time.perf_counter() - start
predictions["Decision Tree"] = dt_pred

def evaluate_model(name, y_true, y_pred, elapsed):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "Cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "Fit_time_sec": elapsed
    }

results.append(evaluate_model("Decision Tree", y_test, dt_pred, fit_times["Decision Tree"]))
display(pd.DataFrame(results).round(4))

In [ ]:
# 19. Random Forest
start = time.perf_counter()
rf_model = RandomForestClassifier(
    n_estimators=120,
    max_depth=20,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf_model.fit(X_train_tree, y_train)
rf_pred = rf_model.predict(X_test_tree)
fit_times["Random Forest"] = time.perf_counter() - start
predictions["Random Forest"] = rf_pred
results.append(evaluate_model("Random Forest", y_test, rf_pred, fit_times["Random Forest"]))

display(pd.DataFrame(results).round(4))

In [ ]:
# 20. LightGBM
from lightgbm import LGBMClassifier

X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()

for c in X_train_lgb.select_dtypes(include=["object"]).columns:
    X_train_lgb[c] = X_train_lgb[c].astype("category")
    # Use the same category set in test.
    X_test_lgb[c] = pd.Categorical(X_test_lgb[c], categories=X_train_lgb[c].cat.categories)

lgb_model = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=250,
    learning_rate=0.06,
    num_leaves=63,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

start = time.perf_counter()
lgb_model.fit(
    X_train_lgb,
    y_train - 1,
    categorical_feature=X_train_lgb.select_dtypes(include=["category"]).columns.tolist()
)
lgb_pred = lgb_model.predict(X_test_lgb).astype(int).ravel() + 1
fit_times["LightGBM"] = time.perf_counter() - start
predictions["LightGBM"] = lgb_pred
results.append(evaluate_model("LightGBM", y_test, lgb_pred, fit_times["LightGBM"]))

results_df = pd.DataFrame(results).sort_values("F1_macro", ascending=False)
display(results_df.round(4))

### Metric rationale

- **Accuracy:** overall proportion of correct predictions.
- **Macro precision/recall/F1:** averages class-level metrics equally, so minority damage grades matter.
- **Weighted F1:** weights each class by its frequency.
- **Balanced accuracy:** average recall across classes.
- **Cohen's kappa:** agreement beyond chance; useful as a supplementary measure.
- **Confusion matrix:** makes the severity of errors visible.

Because grades are ordered, confusing grade 1 with grade 3 is more concerning than confusing grade 1 with grade 2. The confusion matrix should therefore be reviewed alongside the scalar metrics.

In [ ]:
# 21. Classification reports and confusion matrices
for name, pred in predictions.items():
    print("=" * 80)
    print(name)
    print(classification_report(y_test, pred, labels=[1,2,3], digits=4, zero_division=0))

    cm = confusion_matrix(y_test, pred, labels=[1,2,3])
    plt.figure(figsize=(5,4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=[1,2,3], yticklabels=[1,2,3]
    )
    plt.title(f"{name} — Confusion Matrix")
    plt.xlabel("Predicted damage grade")
    plt.ylabel("Actual damage grade")
    plt.tight_layout()
    plt.show()

## 10. Baseline vs Engineered Features

The same LightGBM model configuration is used for a baseline and an engineered-feature version. The comparison is performed on the same held-out test set.

An improvement is reported only if it is actually observed when the notebook runs.

In [ ]:
# 22. LightGBM baseline vs engineered features
def make_lgb_ready(train_x, test_x):
    a, b = train_x.copy(), test_x.copy()
    for c in a.select_dtypes(include=["object"]).columns:
        a[c] = a[c].astype("category")
        b[c] = pd.Categorical(b[c], categories=a[c].cat.categories)
    return a, b

Xb_train, Xb_test = make_lgb_ready(X_train, X_test)
Xe_train, Xe_test = make_lgb_ready(X_train_eng, X_test_eng)

baseline_lgb = LGBMClassifier(
    objective="multiclass", num_class=3,
    n_estimators=250, learning_rate=0.06, num_leaves=63,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)

engineered_lgb = LGBMClassifier(
    objective="multiclass", num_class=3,
    n_estimators=250, learning_rate=0.06, num_leaves=63,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)

t0 = time.perf_counter()
baseline_lgb.fit(Xb_train, y_train - 1, categorical_feature=Xb_train.select_dtypes("category").columns.tolist())
baseline_pred = baseline_lgb.predict(Xb_test).astype(int).ravel() + 1
baseline_time = time.perf_counter() - t0

t0 = time.perf_counter()
engineered_lgb.fit(Xe_train, y_train - 1, categorical_feature=Xe_train.select_dtypes("category").columns.tolist())
engineered_pred = engineered_lgb.predict(Xe_test).astype(int).ravel() + 1
engineered_time = time.perf_counter() - t0

feature_comparison = pd.DataFrame([
    evaluate_model("LightGBM baseline", y_test, baseline_pred, baseline_time),
    evaluate_model("LightGBM engineered", y_test, engineered_pred, engineered_time),
])
display(feature_comparison.round(4))

delta = (
    feature_comparison.loc[feature_comparison["Model"]=="LightGBM engineered","F1_macro"].iloc[0]
    - feature_comparison.loc[feature_comparison["Model"]=="LightGBM baseline","F1_macro"].iloc[0]
)
print(f"Observed macro-F1 change from feature engineering: {delta:+.4f}")

## 11. Cross-Validation

A stratified 3-fold validation is included as an optional robustness check. It is intentionally kept separate from the final held-out test set.

For computational efficiency, the default code below uses LightGBM on the full training partition. If runtime or memory is constrained, set `RUN_CV = False` and report the held-out test result instead of claiming cross-validation evidence.

In [ ]:
# 23. Optional 3-fold CV (disabled by default for faster execution)
RUN_CV = False

if RUN_CV:
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    cv_scores = []
    # Use the already engineered training matrix; each fold still trains independently.
    Xcv = X_train_eng.copy()
    ycv = y_train.copy()

    for fold, (tr_idx, va_idx) in enumerate(cv.split(Xcv, ycv), start=1):
        Xtr_cv = Xcv.iloc[tr_idx].copy()
        Xva_cv = Xcv.iloc[va_idx].copy()
        ytr_cv = ycv.iloc[tr_idx]
        yva_cv = ycv.iloc[va_idx]

        Xtr_cv, Xva_cv = make_lgb_ready(Xtr_cv, Xva_cv)

        model_cv = LGBMClassifier(
            objective="multiclass", num_class=3,
            n_estimators=180, learning_rate=0.07, num_leaves=63,
            class_weight="balanced", random_state=RANDOM_STATE + fold,
            n_jobs=-1, verbosity=-1
        )
        model_cv.fit(
            Xtr_cv, ytr_cv - 1,
            categorical_feature=Xtr_cv.select_dtypes("category").columns.tolist()
        )
        pred_cv = model_cv.predict(Xva_cv).astype(int).ravel() + 1
        score = f1_score(yva_cv, pred_cv, average="macro")
        cv_scores.append(score)
        print(f"Fold {fold} macro F1: {score:.4f}")

    print(f"Mean CV macro F1: {np.mean(cv_scores):.4f}")
    print(f"Std  CV macro F1: {np.std(cv_scores):.4f}")
else:
    print("CV skipped by configuration.")

## 12. Feature Importance

Feature importance is useful for identifying predictive signals, but it should **not** be interpreted as causal evidence. In particular, a geographic identifier can be highly predictive because it captures location-specific construction and hazard context; that does not mean the identifier itself physically causes damage.

For obfuscated categorical variables, only the documented feature name can be discussed. The letters themselves are not assigned semantic meanings.

In [ ]:
# 24. LightGBM feature importance
importance = pd.DataFrame({
    "feature": engineered_lgb.feature_name_,
    "importance_gain": engineered_lgb.booster_.feature_importance(importance_type="gain"),
    "importance_split": engineered_lgb.booster_.feature_importance(importance_type="split")
}).sort_values("importance_gain", ascending=False)

display(importance.head(25))

plt.figure(figsize=(10,7))
top_imp = importance.head(20).sort_values("importance_gain")
plt.barh(top_imp["feature"], top_imp["importance_gain"])
plt.title("Top 20 LightGBM Features by Gain")
plt.xlabel("Total gain")
plt.tight_layout()
plt.show()

In [ ]:
# 25. Random Forest feature importance
feature_names_tree = tree_preprocessor.get_feature_names_out()
rf_imp = pd.DataFrame({
    "feature": feature_names_tree,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

display(rf_imp.head(20))

plt.figure(figsize=(10,7))
top_rf = rf_imp.head(20).sort_values("importance")
plt.barh(top_rf["feature"], top_rf["importance"])
plt.title("Top 20 Random Forest Features")
plt.xlabel("Mean impurity importance")
plt.tight_layout()
plt.show()

## 13. Model Comparison Report

The table below is generated from actual predictions on the independent test partition. The model with the highest macro F1 is not automatically declared universally "best"; computational cost, interpretability, robustness to geographic shift, and intended deployment context must also be considered.

If the highest-scoring model changes when the notebook is rerun with different hyperparameters or a different split, the report should be updated from the measured evidence.

In [ ]:
# 26. Final comparison table
comparison = results_df.copy()
comparison = comparison[
    ["Model","Accuracy","Precision_macro","Recall_macro","F1_macro",
     "F1_weighted","Balanced_accuracy","Cohen_kappa","Fit_time_sec"]
]
display(comparison.round(4))

best_by_f1 = comparison.iloc[comparison["F1_macro"].argmax()]
print("Highest observed macro-F1 model:", best_by_f1["Model"])
print("Its measured macro-F1:", round(best_by_f1["F1_macro"], 4))
print("This is a measured test-set result, not a guarantee of future or global performance.")

### Production-selection framework

Use the measured comparison table to document the selected candidate using these criteria:

1. Macro F1 on the held-out test set.
2. Minority-class recall and confusion patterns.
3. Cross-validation stability, if enabled.
4. Training and inference cost.
5. Interpretability requirements.
6. Sensitivity to geographic distribution shift.
7. Ability to retrain and monitor as new post-disaster data becomes available.

A model trained only on Gorkha earthquake survey data should not be presented as globally validated.

## 14. Challenges Faced and Techniques Used

### Challenge 1 — Large tabular dataset
**Issue:** 260k+ buildings make expensive algorithms and dense one-hot matrices costly.  
**Technique:** Tree-based models and LightGBM are preferred for the main comparison.  
**Reason:** They are appropriate for nonlinear tabular relationships and scale better than kernel methods.

### Challenge 2 — High-cardinality geography
**Issue:** `geo_level_3_id` has thousands of observed regions.  
**Technique:** LightGBM categorical handling plus training-only frequency features.  
**Reason:** Avoids a very large one-hot expansion while retaining geographic information.

### Challenge 3 — Obfuscated categorical values
**Issue:** Category letters do not have documented semantic meaning.  
**Technique:** Treat them as categorical codes only.  
**Reason:** Prevents unsupported assumptions about construction types or legal statuses.

### Challenge 4 — Class imbalance
**Issue:** Damage grade 2 is much more common than grades 1 and 3.  
**Technique:** Stratified split and class-weighted tree/boosting models; macro F1 and balanced accuracy.  
**Reason:** Accuracy alone can hide poor minority-class performance.

### Challenge 5 — Ordinal target
**Issue:** Grades have an order.  
**Technique:** Evaluate as multiclass classification while inspecting the confusion matrix.  
**Reason:** This satisfies the project requirement while making severe 1↔3 errors visible.

### Challenge 6 — Leakage risk
**Issue:** `building_id` is a random unique identifier; target/frequency mappings can leak if computed globally.  
**Technique:** Drop `building_id`; compute learned mappings from training data only.  
**Reason:** Protects the integrity of evaluation.

### Challenge 7 — Geographic generalization
**Issue:** Random splits can contain related buildings/regions in both partitions.  
**Technique:** Explicit limitation and optional future group/geographic validation.  
**Reason:** A stronger real-world test would hold out geographic regions rather than random rows.

## 15. Suggestions to Seismologists and Disaster-Management Stakeholders

The following recommendations should be treated as **risk-reduction hypotheses supported by this dataset's documented variables and observed model associations**, not as engineering conclusions.

1. **Prioritize structural retrofitting programs using evidence from building-level risk models.** Buildings predicted as high-risk can be prioritized for professional inspection and retrofit assessment.
2. **Strengthen seismic-code compliance and construction supervision.** Construction-system and superstructure features are explicit predictors in the dataset and can help identify vulnerable building profiles.
3. **Pay special attention to older buildings.** Age is a documented structural feature and should be considered alongside engineering inspection, material quality, and code compliance.
4. **Use geographic microzonation.** The strong geographic representation in the dataset supports location-aware risk mapping, but geographic identifiers should be validated against geotechnical and seismic-hazard data.
5. **Assess foundations, roofs, ground floors, and plan configuration together.** These are documented building characteristics and should be evaluated as a system rather than as isolated variables.
6. **Promote engineered construction and appropriate seismic detailing.** Any recommendation about a specific construction material should be validated by qualified structural engineers and local seismic codes.
7. **Combine ML with soil testing and hazard maps.** Building vulnerability alone does not fully represent earthquake risk; ground motion, soil conditions, faults, slope stability, and exposure also matter.
8. **Use early-warning and preparedness systems.** Prediction of building vulnerability complements, rather than replaces, earthquake early-warning, evacuation planning, emergency communication, and community preparedness.
9. **Do not automatically demolish or retrofit solely from a model prediction.** Model outputs should trigger professional assessment, not replace it.

## 16. Limitations

- The data represents buildings affected by the 2015 Gorkha earthquake in Nepal; external validity to other countries or earthquake regimes is unknown.
- Categorical values are obfuscated, limiting semantic interpretation.
- Geographic IDs can capture location-specific patterns that may not transfer to unseen regions.
- A random train/test split is not a substitute for a geographic holdout validation.
- Feature importance is associative, not causal.
- Survey data may contain measurement, reporting, or sampling limitations.
- The model does not directly include every physical hazard variable, such as recorded ground acceleration, soil profile, structural calculations, or detailed engineering drawings.
- A high-performing predictive model is not automatically a certified structural-safety tool.

## 17. Future Scope

- Grouped cross-validation by `geo_level_1_id` or `geo_level_2_id`.
- Ordinal classification methods that explicitly model the 1 < 2 < 3 structure.
- Leakage-safe target encoding using nested cross-validation.
- Hyperparameter optimization with Optuna or Bayesian search.
- Calibration of class probabilities.
- SHAP-based local explanations for engineers.
- Integration of seismic intensity, soil, slope, fault-distance, and remote-sensing data.
- External validation on another earthquake event before any operational deployment.
- Monitoring for geographic and temporal distribution shift.

## 18. Conclusion

This notebook provides a reproducible end-to-end workflow for PRCP-1015: verifying the earthquake-damage dataset, analyzing its structural and geographic characteristics, preprocessing mixed tabular data, engineering leakage-aware features, comparing multiple machine-learning models, and evaluating predictions with macro F1 and complementary metrics.

The final model recommendation must be taken from the **measured comparison table produced during execution**. The notebook intentionally avoids fabricated accuracy or F1 values and keeps the distinction between predictive association and engineering causation.

The model should be used as an academic decision-support component and validated by domain experts before any real-world structural or disaster-management application.

## 19. Reproducibility / Execution Guide

### Recommended environment
- Python 3.9+
- Jupyter Notebook or JupyterLab

### Install dependencies
```bash
pip install pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost catboost
```

### Dataset placement
Put these files in the same directory as the notebook, or change `DATA_DIR`:
- `train_values.csv`
- `train_labels.csv`

The loader also supports a `train.csv` containing the target if adapted accordingly.

### Run
1. Open `PRCP-1015_Earthquake_Damage_Prediction.ipynb`.
2. Set `DATA_DIR`.
3. Run cells from top to bottom.
4. Review the actual dataset verification output before modeling.
5. Save the executed notebook for academic submission.

### Memory/runtime guidance
- Avoid converting the entire high-cardinality geographic feature set to dense one-hot matrices.
- LightGBM is used because it is efficient for large tabular data.
- If RAM is limited, reduce Random Forest `n_estimators` or run LightGBM first.
- If cross-validation is too slow, set `RUN_CV = False` and clearly document that it was skipped.

### Important reproducibility note
The notebook fixes `RANDOM_STATE = 42` for the main split and stochastic models. Results can still vary across library versions and hardware.

## References

1. DrivenData — *Richter's Predictor: Modeling Earthquake Damage*.  
   https://www.drivendata.org/competitions/57/nepal-earthquake/

2. Project requirements document supplied for PRCP-1015.

3. Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5–32.

4. Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *Advances in Neural Information Processing Systems*.

5. Scikit-learn documentation — model evaluation and preprocessing.